In [0]:
 %run "/Workspace/Users/jeancarlosramoshuamanlazo@hotmail.com/databricksjc/Proceso/0_CONFIGURACION"



In [0]:
from pyspark.sql import functions as F

spark.sql(f"USE CATALOG {catalog_name}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {silver_catalog}")

In [0]:
df = spark.table(bronze_tables["automobile"])

In [0]:
def clean_string(col):
    return F.trim(F.lower(F.col(col)))

def safe_double(col):
    return F.col(col).cast("double")

def safe_int(col):
    return F.col(col).cast("int")

audit_cols = ["_ingestion_ts", "_record_hash", "_source_file", "_load_date"]

AUTOS_CLEAN

In [0]:
autos_clean = (
    df
    .select(
        clean_string("make").alias("make"),
        clean_string("fuel_type").alias("fuel_type"),
        clean_string("aspiration").alias("aspiration"),
        clean_string("body_style").alias("body_style"),
        clean_string("drive_wheels").alias("drive_wheels"),

        safe_double("price").alias("price"),
        safe_double("horsepower").alias("horsepower"),
        safe_int("city_mpg").alias("city_mpg"),
        safe_int("highway_mpg").alias("highway_mpg"),

        *[c for c in audit_cols if c in df.columns]
    )
    .withColumn(
        "record_status",
        F.when(F.col("price").isNull(), F.lit("INVALID"))
         .otherwise(F.lit("VALID"))
    )
    .dropDuplicates()
)

save_delta_table(autos_clean, silver_tables["autos_clean"])

ENGINE-SPECS

In [0]:
engine_specs = (
    df
    .select(
        clean_string("engine_type").alias("engine_type"),
        clean_string("num_of_cylinders").alias("num_of_cylinders"),

        safe_int("engine_size").alias("engine_size"),
        safe_double("bore").alias("bore"),
        safe_double("stroke").alias("stroke"),
        safe_double("compression_ratio").alias("compression_ratio"),
        safe_double("horsepower").alias("horsepower"),

        clean_string("fuel_system").alias("fuel_system"),

        *[c for c in audit_cols if c in df.columns]
    )
    .dropDuplicates()
)

save_delta_table(engine_specs, silver_tables["engine_specs"])

VEHICULE_SPECS

In [0]:
vehicle_specs = (
    df
    .select(
        safe_double("length").alias("length"),
        safe_double("width").alias("width"),
        safe_double("height").alias("height"),
        safe_int("curb_weight").alias("curb_weight"),
        safe_double("wheel_base").alias("wheel_base"),

        *[c for c in audit_cols if c in df.columns]
    )
    .dropDuplicates()
)

save_delta_table(vehicle_specs, silver_tables["vehicle_specs"])

ANALISIS AUTOS_ENRICHED

In [0]:
autos_enriched = (
    autos_clean.alias("a")
    .join(
        engine_specs.alias("e"),
        F.col("a.horsepower") == F.col("e.horsepower"),
        "left"
    )
    .select(
        "a.*",
        F.col("e.engine_type"),
        F.col("e.num_of_cylinders"),
        F.col("e.engine_size"),
        F.col("e.fuel_system")
    )
)

In [0]:

autos_conformed = (
    df
    .select(
        clean_string("make").alias("make"),
        clean_string("fuel_type").alias("fuel_type"),
        clean_string("aspiration").alias("aspiration"),
        clean_string("body_style").alias("body_style"),
        clean_string("drive_wheels").alias("drive_wheels"),

        clean_string("engine_type").alias("engine_type"),
        clean_string("num_of_cylinders").alias("num_of_cylinders"),
        safe_int("engine_size").alias("engine_size"),
        clean_string("fuel_system").alias("fuel_system"),

        safe_double("price").alias("price"),
        safe_double("horsepower").alias("horsepower"),
        safe_int("city_mpg").alias("city_mpg"),
        safe_int("highway_mpg").alias("highway_mpg"),
        safe_int("curb_weight").alias("curb_weight"),

        *[c for c in audit_cols if c in df.columns]
    )
    .dropDuplicates()
)

save_delta_table(autos_conformed, f"{silver_catalog}.autos_conformed")